In [95]:
import os
import sys
import warnings

In [96]:
from pyspark.sql import SparkSession
from pyspark import SparkContext,SparkConf

In [97]:
conf = SparkConf().setAppName("Lab2").setMaster("local[*]")

In [98]:
sc = SparkContext.getOrCreate(conf = conf)
sc.setLogLevel("Error")

In [99]:
print("Spark.version: ",sc.version)
print("App Name :",sc.appName)

Spark.version:  3.5.0
App Name : Lab2


In [100]:
warnings.filterwarnings('ignore')

In [101]:
numbers = [10,5,4,45,67,23,45,32,13,67,86, 78,45,98,23,42,61]
rdd_nums = sc.parallelize(numbers)
print("Type: ",type(rdd_nums))
print("Partition counts :",rdd_nums.getNumPartitions())
print("Count :",rdd_nums.count())
print("First 5 counts : ",rdd_nums.take(5))

Type:  <class 'pyspark.rdd.RDD'>
Partition counts : 8


Count : 17
First 5 counts :  [10, 5, 4, 45, 67]


In [102]:
employees = [
    "Ram,Doctor,12000",
    "Hari,Peon,10000",
    "John,Cleaner,23000",
    "Marissa,Volunteer,45000",
    "Ramesh,Doctor,140000",
    "Hardik,Lawyer,10000",
    "Josh,Bookkeeper,23000",
    "Julie,Volunteer,4000",
]

In [103]:
rdd_emp = sc.parallelize(employees)
print("Employee count: ",rdd_emp.count())
print("First record : ",rdd_emp.first())




Employee count:  8
First record :  Ram,Doctor,12000


In [104]:
rdd_range = sc.range(1,21)
print(rdd_range)


PythonRDD[86] at RDD at PythonRDD.scala:53


In [105]:
rdd_range.collect()



[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

In [106]:
print("range sum :",rdd_range.sum())


range sum : 210


In [107]:
import shutil
emp_path = "lab_output/employees"

if os.path.exists(emp_path):
    shutil.rmtree(emp_path)


In [108]:
rdd_emp.saveAsTextFile(emp_path)
print("Employees saved to :",emp_path)

Employees saved to : lab_output/employees


In [109]:
with open("lab_output/employees/part-00000", "r") as f:
    print(f.read())

Ram,Doctor,12000



In [110]:
files = os.listdir(emp_path)
partition_files = [f for f in files if f.startswith("part")]
print("Files written : ",files)
print("Number of part files : ",len(partition_files))

Files written :  ['part-00002', 'part-00006', '._SUCCESS.crc', 'part-00000', '.part-00003.crc', '.part-00006.crc', '.part-00001.crc', '.part-00004.crc', 'part-00007', '_SUCCESS', 'part-00003', '.part-00000.crc', 'part-00005', 'part-00004', '.part-00005.crc', '.part-00007.crc', '.part-00002.crc', 'part-00001']
Number of part files :  8


In [111]:
rdd_emp_loaded = sc.textFile("lab_output/employees/")
rdd_emp_loaded.collect()


['John,Cleaner,23000',
 'Josh,Bookkeeper,23000',
 'Ram,Doctor,12000',
 'Julie,Volunteer,4000',
 'Marissa,Volunteer,45000',
 'Hardik,Lawyer,10000',
 'Ramesh,Doctor,140000',
 'Hari,Peon,10000']

In [112]:
def parse_row(line):
    parts = line.split(",")
    name = parts[0]
    dept = parts[1]
    salary = int(parts[2])
    return (name,dept,salary)
    

In [113]:
rdd_parsed = rdd_emp_loaded.map(parse_row)
rdd_parsed.collect()

[('John', 'Cleaner', 23000),
 ('Josh', 'Bookkeeper', 23000),
 ('Ram', 'Doctor', 12000),
 ('Julie', 'Volunteer', 4000),
 ('Marissa', 'Volunteer', 45000),
 ('Hardik', 'Lawyer', 10000),
 ('Ramesh', 'Doctor', 140000),
 ('Hari', 'Peon', 10000)]

In [116]:
# .strip(" '\"") removes spaces, single quotes, and double quotes from both ends
rdd_doc = rdd_parsed.filter(lambda r: r[1] == "Doctor" and r[2]>40000)

print("Doctor count: ", rdd_doc.count())

Doctor count:  1


In [117]:
# 2. Map to extract just the name (assuming index 0 is the name)
rdd_doc_names = rdd_doc.map(lambda r: r[0])

# 3. Collect and print the names
doctor_names = rdd_doc_names.collect()

print(f"Found {len(doctor_names)} doctors:")
for name in doctor_names:
    print(f"- {name}")

Found 1 doctors:
- Ramesh
